# 15 — Evaluation: Correctness, Grounding, Safety, and Operations

**Network LLM Engineering — Part III — Adaptation**

### Learning goals
- Build a multi-dimensional evaluation harness
- Separate capability from safety and operations
- Use held-out network cases and deterministic checks

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## One score is not enough

A Network LLM release should measure at least:

**Capability**
- routing/protocol correctness
- troubleshooting coverage
- config interpretation
- schema adherence

**Grounding**
- claims supported by supplied telemetry/docs
- no invented observations

**Safety**
- verify before change
- privilege boundaries
- prompt injection resistance

**Operations**
- latency
- throughput
- token usage/cost
- tool failure handling
- trace completeness

In [ ]:
import ipaddress, json

def subnet_exact_score(answer_json, cidr):
    try:
        a = json.loads(answer_json) if isinstance(answer_json, str) else answer_json
        n = ipaddress.ip_network(cidr, strict=False)
        expected = {
            "network": str(n.network_address),
            "broadcast": str(n.broadcast_address),
            "prefixlen": n.prefixlen,
            "usable_hosts": max(0, n.num_addresses - 2),
        }
        return int(all(a.get(k)==v for k,v in expected.items()))
    except Exception:
        return 0

print(subnet_exact_score(
    '{"network":"192.0.2.64","broadcast":"192.0.2.127","prefixlen":26,"usable_hosts":62}',
    "192.0.2.64/26"))

## LLM-as-judge caution

An LLM judge is useful for scalable qualitative grading, but it is itself a model with biases and failure modes.
Calibrate it against expert human labels. For verifiable tasks, deterministic scoring is stronger.

## Golden set

Maintain a small expert-reviewed release gate covering:
- healthy and broken states,
- multiple vendors/topologies,
- ambiguous evidence,
- adversarial instructions,
- tool outages,
- general-knowledge regressions.

## Optional: evaluate the LoRA adapter from Notebook 13

If you ran the LoRA lab, use the course's held-out networking challenges to compare the **same prompts**
against the base model and adapted model. This prevents "the training loss went down" from being mistaken for success.

In [ ]:
from pathlib import Path
adapter_dir = ROOT / "artifacts" / "network-lora"
challenge_file = DATA / "network_challenges.jsonl"

if adapter_dir.exists() and challenge_file.exists():
    import torch, json, pandas as pd
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    model_id = "Qwen/Qwen3-0.6B"
    tok = AutoTokenizer.from_pretrained(adapter_dir)
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
        torch.float16 if torch.cuda.is_available() else torch.float32
    )

    def ask(m, q):
        messages = [
            {"role":"system","content":"You are a careful network troubleshooting assistant. Verify evidence before changes."},
            {"role":"user","content":q},
        ]
        try:
            prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        x = tok(prompt, return_tensors="pt")
        dev = next(m.parameters()).device
        x = {k:v.to(dev) for k,v in x.items()}
        with torch.inference_mode():
            y = m.generate(**x, max_new_tokens=160, do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(y[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def coverage(text, keywords):
        t = text.lower()
        return sum(k.lower() in t for k in keywords) / max(1, len(keywords))

    challenges = [json.loads(x) for x in open(challenge_file, encoding="utf-8")]

    base = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None
    ).eval()
    base_rows = []
    for item in challenges:
        a = ask(base, item["prompt"])
        base_rows.append((item["id"], "base", coverage(a, item["expected_keywords"]), a))
    del base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    b = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None
    )
    tuned = PeftModel.from_pretrained(b, adapter_dir).eval()
    tuned_rows = []
    for item in challenges:
        a = ask(tuned, item["prompt"])
        tuned_rows.append((item["id"], "lora", coverage(a, item["expected_keywords"]), a))

    df = pd.DataFrame(base_rows + tuned_rows, columns=["id","model","keyword_coverage","answer"])
    display(df.groupby("model")["keyword_coverage"].mean())
    display(df.pivot(index="id", columns="model", values="keyword_coverage"))
else:
    print("Run Notebook 13 first (and ensure network_challenges.jsonl is present) to enable this comparison.")

### Exercise

Design a 20-case golden set for one technology (EVPN, ACI, SD-WAN, ISE, etc.).
Define a rubric **before** tuning against it.